## Partie 1 – Configuration

In [1]:
%%writefile src/collecte.py
"""Collecte des disponibilités Vélib' et envoi vers BigQuery."""
import logging
from pathlib import Path

import pandas as pd
import pandas_gbq
import requests

PROJET = "velocite-velib"
DATASET = "velib"
URL_INFO = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
URL_STATUS = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

Overwriting src/collecte.py


## Partie 2 – Traces d'exécution (logs)

In [2]:
%%writefile -a src/collecte.py


DOSSIER_LOGS = Path(__file__).resolve().parent.parent / "logs"
DOSSIER_LOGS.mkdir(exist_ok=True)
logging.basicConfig(
    filename=DOSSIER_LOGS / "collecte.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8",
)

Appending to src/collecte.py


## Partie 3 – Appel de l'API

In [3]:
%%writefile -a src/collecte.py


def appeler_api(url):
    """Appelle l'API et renvoie les stations sous forme de DataFrame."""
    reponse = requests.get(url, timeout=30)
    reponse.raise_for_status()
    return pd.DataFrame(reponse.json()["data"]["stations"])

Appending to src/collecte.py


## Partie 4 – Table stations

In [4]:
%%writefile -a src/collecte.py


def preparer_stations(df_info):
    """Table stations : infos fixes de chaque station."""
    df_info["accepte_cb"] = df_info["rental_methods"].notna()
    colonnes = ["station_id", "stationCode", "name", "lat", "lon", "capacity", "accepte_cb"]
    return df_info[colonnes].drop_duplicates(subset="station_id")

Appending to src/collecte.py


## Partie 5 – Table disponibilites

In [5]:
%%writefile -a src/collecte.py


def preparer_disponibilites(df_status, df_stations):
    """Table disponibilites : une photo de chaque station par heure."""
    df = df_status.copy()
    df["nb_mecaniques"] = df["num_bikes_available_types"].str[0].str["mechanical"]
    df["nb_electriques"] = df["num_bikes_available_types"].str[1].str["ebike"]

    df["last_reported"] = pd.to_datetime(df["last_reported"], unit="s", utc=True)
    df["date_collecte"] = pd.Timestamp.now(tz="UTC")
    # Clé unique : station + heure de collecte (ex. 213688169_2026101309)
    df["cle"] = df["station_id"].astype(str) + "_" + df["date_collecte"].dt.strftime("%Y%m%d%H")

    df = df.merge(df_stations[["station_id", "capacity"]], on="station_id", how="left")
    df = df[(df["capacity"] > 0) & (df["is_installed"] == 1)]
    df["taux_remplissage"] = df["num_bikes_available"] / df["capacity"]
    df["station_vide"] = df["num_bikes_available"] == 0
    df["station_pleine"] = df["num_docks_available"] == 0

    colonnes = ["cle", "station_id", "num_bikes_available", "num_docks_available",
                "nb_mecaniques", "nb_electriques", "is_renting", "is_returning",
                "last_reported", "date_collecte", "taux_remplissage",
                "station_vide", "station_pleine"]
    return df[colonnes].drop_duplicates(subset="cle")

Appending to src/collecte.py


## Partie 6 – Anti-doublons

In [6]:
%%writefile -a src/collecte.py


def retirer_deja_presentes(df):
    """Anti-doublons : on retire les lignes dont la clé est déjà dans BigQuery."""
    requete = f"SELECT cle FROM `{PROJET}.{DATASET}.disponibilites`"
    try:
        deja = pandas_gbq.read_gbq(requete, project_id=PROJET)
    except Exception:
        logging.info("Table disponibilites absente : première collecte")
        return df
    return df[~df["cle"].isin(deja["cle"])]

Appending to src/collecte.py


## Partie 7 – Journal des exécutions

In [7]:
%%writefile -a src/collecte.py


def ecrire_journal(statut, nb_lignes, message):
    """Trace de chaque exécution dans la table journal_executions."""
    journal = pd.DataFrame([{
        "date_execution": pd.Timestamp.now(tz="UTC"),
        "statut": statut,
        "nb_lignes": nb_lignes,
        "message": message,
    }])
    pandas_gbq.to_gbq(journal, f"{DATASET}.journal_executions", project_id=PROJET, if_exists="append")

Appending to src/collecte.py


## Partie 8 – Programme principal

In [8]:
%%writefile -a src/collecte.py


def main():
    logging.info("Début de la collecte")
    try:
        df_stations = preparer_stations(appeler_api(URL_INFO))
        df_dispo = preparer_disponibilites(appeler_api(URL_STATUS), df_stations)
        nb_recues = len(df_dispo)
        df_dispo = retirer_deja_presentes(df_dispo)

        pandas_gbq.to_gbq(df_stations, f"{DATASET}.stations", project_id=PROJET, if_exists="replace")
        if len(df_dispo) > 0:
            pandas_gbq.to_gbq(df_dispo, f"{DATASET}.disponibilites", project_id=PROJET, if_exists="append")

        message = f"{nb_recues} lignes reçues, {len(df_dispo)} nouvelles envoyées"
        logging.info(message)
        ecrire_journal("OK", len(df_dispo), message)
    except Exception as erreur:
        logging.error(f"Échec de la collecte : {erreur}")
        try:
            ecrire_journal("ERREUR", 0, str(erreur))
        except Exception:
            pass
        raise


if __name__ == "__main__":
    main()

Appending to src/collecte.py


## Test du script

In [9]:
!python src/collecte.py


Downloading:   0%|          |
Downloading: 100%|##########|



100%|##########| 1/1 [00:00<00:00, 1460.41it/s]

100%|##########| 1/1 [00:00<00:00, 943.18it/s]


In [10]:
print(open("logs/collecte.log", encoding="utf-8").read())

2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-

In [11]:
%%writefile lancer_collecte.bat
@echo off
cd /d "C:\Users\utilisateur\velib suz"
call "C:\Users\utilisateur\anaconda3\Scripts\activate.bat"
python src\collecte.py

Overwriting lancer_collecte.bat


In [12]:
!schtasks /create /tn "Collecte Velib" /tr "\"C:\Users\utilisateur\velib suz\lancer_collecte.bat\"" /sc hourly /f

Opération réussie : la tâche planifiée "Collecte Velib" a été créée.


In [15]:
!schtasks /run /tn "Collecte Velib"

Opération réussie : tentative d’exécution de la tâche planifiée "Collecte Velib".


In [16]:
print(open("logs/collecte.log", encoding="utf-8").read())


2026-10-13 10:20:32,854 | INFO | Début de la collecte
2026-10-13 10:20:37,675 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:20:41,918 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:20:44,240 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:20:47,577 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:21,774 | INFO | Début de la collecte
2026-10-13 10:22:26,230 | INFO | Table disponibilites absente : première collecte
2026-10-13 10:22:29,157 | INFO | 
1519 out of 1519 rows loaded.
2026-10-13 10:22:30,505 | ERROR | Échec de la collecte : Reason: Provided Schema does not match Table velocite-velib:velib.disponibilites. Cannot add fields (field: cle)
2026-10-13 10:22:33,686 | INFO | 
1 out of 1 rows loaded.
2026-10-13 10:22:37,092 | INFO | Début de la collecte
2026-10-13 10:22:41,598 | INFO | Table disponibilites absente : première collecte
2026-10-

## Requêtes SQL

In [17]:
import os
os.makedirs("sql", exist_ok=True)

### Contrôle qualité : doublons

In [18]:
%%writefile sql/00_controle_doublons.sql
-- Contrôle qualité : vérifie qu'il n'y a aucun doublon
-- Si nb_lignes = nb_cles_uniques, chaque photo (station + heure) est unique
SELECT
  COUNT(*) AS nb_lignes,
  COUNT(DISTINCT cle) AS nb_cles_uniques
FROM `velocite-velib.velib.disponibilites`

Writing sql/00_controle_doublons.sql


### Requête 1 – Stations vides maintenant

In [19]:
%%writefile sql/01_stations_vides_maintenant.sql
-- Requête 1 : stations vides lors de la dernière collecte, les plus grandes d'abord
SELECT s.name, s.stationCode, s.capacity, d.date_collecte
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
WHERE d.station_vide
  AND d.date_collecte = (SELECT MAX(date_collecte) FROM `velocite-velib.velib.disponibilites`)
ORDER BY s.capacity DESC

Writing sql/01_stations_vides_maintenant.sql


### Requête 2 – Situation par heure

In [20]:
%%writefile sql/02_situation_par_heure.sql
-- Requête 2 : situation moyenne par heure (heure de Paris)
SELECT
  EXTRACT(HOUR FROM DATETIME(date_collecte, "Europe/Paris")) AS heure,
  ROUND(100 * AVG(taux_remplissage), 1) AS taux_remplissage_moyen_pct,
  ROUND(100 * COUNTIF(station_vide) / COUNT(*), 1) AS pct_stations_vides,
  ROUND(100 * COUNTIF(station_pleine) / COUNT(*), 1) AS pct_stations_pleines
FROM `velocite-velib.velib.disponibilites`
GROUP BY heure
ORDER BY heure

Writing sql/02_situation_par_heure.sql


### Requête 3 – Top 10 des stations le plus souvent vides

In [21]:
%%writefile sql/03_top10_stations_vides.sql
-- Requête 3 : top 10 des stations le plus souvent vides
SELECT
  s.station_id, s.name, s.stationCode,
  COUNT(*) AS nb_photos,
  COUNTIF(d.station_vide) AS nb_fois_vide,
  ROUND(100 * COUNTIF(d.station_vide) / COUNT(*), 1) AS pct_temps_vide
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
GROUP BY s.station_id, s.name, s.stationCode
ORDER BY pct_temps_vide DESC, nb_fois_vide DESC
LIMIT 10

Writing sql/03_top10_stations_vides.sql


## Publication sur GitHub

In [22]:
!git init
!git branch -M main

Initialized empty Git repository in C:/Users/utilisateur/velib suz/.git/


In [23]:
!git config user.name "suzy5670"
!git config user.email "mdane230@gmail.com"

In [24]:
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/
	collecte.py/
	lancer_collecte.bat
	logs/
	sql/
	src/
	velib.ipynb
	velib2.ipynb

nothing added to commit but untracked files present (use "git add" to track)


## Préparation de la publication sur GitHub
Avant d'envoyer le projet, on définit ce qui ne doit **jamais** être publié et les bibliothèques nécessaires pour que le projet puisse être repris par une autre personne.

### Fichier `.gitignore`
Liste des fichiers ignorés par Git : les secrets (`*.json`, `.env`), les logs et les fichiers temporaires de Jupyter.

In [25]:
%%writefile .gitignore
# Secrets : ne jamais publier
*.json
.env

# Fichiers générés automatiquement
logs/
__pycache__/
.ipynb_checkpoints/

Writing .gitignore


### Fichier `requirements.txt`
Bibliothèques Python du projet, avec leurs versions. Installation : `pip install -r requirements.txt`.

In [26]:
%%writefile requirements.txt
requests==2.34.2
pandas==2.3.3
pandas-gbq==0.35.2

Writing requirements.txt


### Nettoyage
Suppression d'un dossier vide `collecte.py` créé par erreur à la racine (le vrai script est dans `src/`).

In [27]:
import shutil, os

if os.path.isdir("collecte.py"):
    shutil.rmtree("collecte.py")
    print("Faux dossier supprimé ✅")
else:
    print("Rien à supprimer")

Faux dossier supprimé ✅


### Vérification avant envoi
On vérifie qu'aucun fichier sensible (logs, clés) ne fait partie des fichiers à publier.